# MobileNetV3 Large — Face Recognition Training

**Mục tiêu:** Train student model đơn giản (single-task, không FSM) cho face recognition.  
**Pipeline tiếp theo:** Knowledge Distillation từ ConvNeXt V2 → Quantize INT8 → Deploy edge device.

| Thành phần | Chi tiết |
|---|---|
| Backbone | MobileNetV3 Large (pretrained ImageNet) |
| Embedding | 512-D (BN → AdaptiveAvgPool → Flatten) |
| Loss | MagFace (class-balanced, adaptive margin) |
| Sampler | PK Sampler — mỗi batch có `batch_size` identity khác nhau |
| Metric | AUC cosine similarity + AUC euclidean distance |

## 1. Setup môi trường

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Clone repo (chỉ chạy lần đầu)
REPO_URL    = 'https://github.com/minh4923/FR_Photometric_Stereo.git'
REPO_BRANCH = 'main'   # <-- đổi sang branch của bạn nếu cần
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL}

%cd {REPO_DIR}

# Cài thư viện bổ sung
!pip install -q albumentations==1.3.1 timm tabulate termcolor

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from going_modular.dataloader.multitask import create_multitask_datafetcher
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.FaceRecognitionLoss import FaceRecognitionLoss
from going_modular.train_eval.fr_train import fit
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
#  CẤU HÌNH — chỉnh sửa ở đây
# ============================================================
EXPERIMENT_NAME = 'MobileNetV3_Albedo_PK'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,

    # Đường dẫn dataset trên Google Drive
    'dataset_dir': '/content/drive/MyDrive/Photometric_DB_Full/',

    # Modality: 'albedo' | 'normalmap' | 'depthmap'
    'type':        'albedo',

    # Backbone timm name
    'backbone':    'mobilenetv3_large_100',

    # PK Sampler: True = mỗi batch gồm batch_size identity khác nhau (tốt hơn cho metric learning)
    'use_sampler': True,

    'device':      device,
    'epochs':      100,
    'batch_size':  32,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,   # tự động lấy từ CSV
}

## 3. Data Loading

In [ ]:
# Đọc CSV để lấy num_classes
train_csv = os.path.join(CONFIGURATION['dataset_dir'], 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(CONFIGURATION['dataset_dir'], 'dataset', 'train_split.csv')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f"Số identity (num_classes): {CONFIGURATION['num_classes']}")
print(f"Số mẫu train: {len(df_train)}")

# Augmentation
train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])

test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform
)
print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

## 4. Model, Loss, Optimizer

In [ ]:
# Model
model = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
model.to(device)

# Loss — MagFace với class-balanced weighting
criterion = FaceRecognitionLoss(metadata_path=train_csv)

# Optimizer — tất cả params (backbone + embedding + maglinear + log_vars)
optimizer = Adam(model.parameters(), lr=CONFIGURATION['base_lr'])

# Scheduler — cosine annealing với warm restart
scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-6
)

# Thống kê model
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## 5. Training

In [ ]:
# Experiment Manager — tạo thư mục log + checkpoint
manager = ExperimentManager(CONFIGURATION)
manager.log_text(f"Backbone: {CONFIGURATION['backbone']} | Modality: {CONFIGURATION['type']} | PK Sampler: {CONFIGURATION['use_sampler']}")

# Checkpoint — lưu best model theo AUC cosine
ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)

# Early Stopping
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=20,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

In [ ]:
fit(
    conf=CONFIGURATION,
    start_epoch=0,
    model=model,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    early_stopping=early_stopping,
    model_checkpoint=ckpt_saver,
    existing_manager=manager,
)

## 6. Resume Training từ Checkpoint

> Chạy cell này khi muốn tiếp tục train từ lần dừng trước.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')  # hoặc 'best_model.pth'

checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

start_epoch = checkpoint['epoch']
print(f"Resume từ epoch {start_epoch}")

fit(
    conf=CONFIGURATION,
    start_epoch=start_epoch,
    model=model,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    early_stopping=early_stopping,
    model_checkpoint=ckpt_saver,
    existing_manager=manager,
)

## 7. Đánh giá & Trích xuất Embedding

> Load best model và tính AUC trên tập test.

In [ ]:
from going_modular.utils.roc_auc_id import compute_id_auc

# Load best model
best_ckpt = torch.load(os.path.join(manager.ckpt_dir, 'best_model.pth'), map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

# Tính AUC
train_auc = compute_id_auc(train_dl, model, device)
test_auc  = compute_id_auc(test_dl,  model, device)

print(f"Train — Cosine AUC: {train_auc['id_cosine']:.4f} | Euclidean AUC: {train_auc['id_euclidean']:.4f}")
print(f"Test  — Cosine AUC: {test_auc['id_cosine']:.4f}  | Euclidean AUC: {test_auc['id_euclidean']:.4f}")

In [ ]:
# Export backbone + embedding (bỏ MagLinear) sang ONNX để chuẩn bị quantize
import torch.nn as nn

class InferenceModel(nn.Module):
    """Chỉ giữ phần inference: backbone + embedding (không có MagLinear)."""
    def __init__(self, fr_model):
        super().__init__()
        self.backbone  = fr_model.backbone
        self.embedding = fr_model.embedding

    def forward(self, x):
        import torch.nn.functional as F
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb, p=2, dim=1)  # L2 normalized

inference_model = InferenceModel(model).eval().to('cpu')

dummy_input = torch.randn(1, 3, 112, 112)
onnx_path   = os.path.join(manager.ckpt_dir, 'mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')